<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2007%20-%202026-04-29%20-%20Pandas%20II/clase_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 7 · Pandas II · Limpieza y transformación de datos

**Fecha:** Miércoles 29 de abril de 2026  
**Módulo:** Módulo 2 · Manejo de datos y estadística  

---

## Introducción

La mayor parte del tiempo en un proyecto de datos se va en limpiar. Hoy veremos manejo de nulos, duplicados, cambio de tipos, outliers, `apply`, `map`, `merge` y `groupby`.

## 1. Por qué limpiar · el 80% del trabajo real

Según encuestas repetidas de Kaggle, Forbes y O'Reilly, los analistas y científicos de datos dedican entre **60% y 80% de su tiempo a limpiar y preparar datos**. No es algo que se pueda saltar: un modelo entrenado sobre datos sucios devuelve predicciones sucias. El mismo Andrew Ng popularizó el término *data-centric AI* en 2021 para enfatizar que mejorar los datos suele rendir más que cambiar el modelo.

Los problemas típicos:

- **Valores faltantes** (nulos, `NaN`, `None`, celdas vacías).
- **Duplicados** (filas idénticas o casi idénticas).
- **Tipos incorrectos** (fechas guardadas como string, números con coma como decimal).
- **Outliers** (valores extremos legítimos o errores de captura).
- **Inconsistencias** (mismo dato escrito de varias formas: "Quito", "quito", "QUITO ", "UIO").

## 2. Valores faltantes · tres tipos

Los estadísticos Rubin y Little clasificaron los nulos en 3 categorías. La clasificación importa porque determina cómo tratarlos:

| Tipo | Significa | Tratamiento razonable |
|---|---|---|
| **MCAR** (*Missing Completely At Random*) | Los nulos son aleatorios, sin relación con nada | Eliminar o imputar |
| **MAR** (*Missing At Random*) | Los nulos dependen de **otras variables observadas** | Imputar condicionado (por grupo) |
| **MNAR** (*Missing Not At Random*) | Los nulos dependen del **propio valor faltante** | Modelar el proceso de faltante |

Ejemplo: si en una encuesta de salario los que ganan más deciden no responder, es **MNAR** y eliminar esas filas sesga el análisis a la baja.

**Reglas prácticas**:
- Si una columna tiene >40% de nulos, suele ser mejor descartarla.
- Si una fila tiene >50% de nulos, eliminarla.
- Imputar con la mediana (numéricos) o la moda (categóricos) **por grupo** suele funcionar mejor que imputar globalmente.

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
print(df.isna().sum())

In [ ]:
# Imputar edad con la mediana por clase (MAR: la edad puede depender de la clase)
df["Age"] = df.groupby("Pclass")["Age"].transform(lambda s: s.fillna(s.median()))

# Eliminar filas sin Embarked (solo 2 nulos, no vale la pena imputar)
df = df.dropna(subset=["Embarked"])

print("Nulos restantes:")
print(df.isna().sum())

## 3. Duplicados

`df.duplicated()` devuelve una Serie booleana marcando duplicados. `df.drop_duplicates()` los elimina. Argumentos importantes:

- `subset=[...]` — considera solo esas columnas para detectar duplicados.
- `keep="first"` / `"last"` / `False` — cuál de los duplicados conservar (o ninguno).

Ojo: **no siempre hay que eliminarlos**. Si el dataset son transacciones, que dos filas sean iguales puede ser real (mismo precio, mismo día, mismo producto).

In [ ]:
print("Duplicados:", df.duplicated().sum())
df = df.drop_duplicates()

## 4. Tipos de datos

Convertir tipos tiene dos beneficios: **ahorra memoria** y **habilita operaciones**. Las conversiones más útiles:

- `astype("category")` — para columnas con pocos valores únicos. Ahorra memoria drásticamente.
- `astype(bool)` — para columnas 0/1 que representan sí/no.
- `pd.to_datetime(col)` — convierte strings a fechas. Habilita `.dt.year`, `.dt.month`, etc.
- `pd.to_numeric(col, errors="coerce")` — fuerza a número, convirtiendo errores en `NaN`.

In [ ]:
df["Pclass"] = df["Pclass"].astype("category")
df["Survived"] = df["Survived"].astype(bool)
df.dtypes

## 5. Outliers · la regla del IQR de Tukey

Un **outlier** es un valor atípico respecto al resto. Detectarlos importa porque distorsionan promedios, desviaciones y modelos. La regla más conocida la propuso **John Tukey** en 1977:

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1a/Boxplot_vs_PDF.svg" width="420" alt="Boxplot y distribución">

Calcular el **rango intercuartílico** `IQR = Q3 - Q1`. Todo valor por debajo de `Q1 - 1.5·IQR` o por encima de `Q3 + 1.5·IQR` se marca como outlier.

No todos los outliers son errores. En finanzas, los grandes movimientos son la parte interesante. Regla: **documenten la decisión** — ¿los eliminan, los capan, los dejan?

In [ ]:
q1 = df["Fare"].quantile(0.25)
q3 = df["Fare"].quantile(0.75)
iqr = q3 - q1
umbral_alto = q3 + 1.5 * iqr
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr, "| Umbral alto:", umbral_alto)
print("Outliers en Fare:", (df["Fare"] > umbral_alto).sum())

## 6. `apply` y `map` · transformaciones personalizadas

Cuando una transformación no existe como método de pandas, `apply` aplica una función arbitraria a cada fila o columna:

- `serie.apply(f)` — aplica `f` a cada elemento de la Series.
- `df.apply(f)` — aplica `f` a cada columna (o fila con `axis=1`).
- `serie.map({"a": 1, "b": 2})` — mapeo directo con un diccionario.

Para operaciones simples elemento a elemento, **usen operaciones vectorizadas** (`serie * 2`, `serie.str.upper()`). `apply` solo es necesario cuando la lógica es más compleja.

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Extraer el título (Mr, Mrs, Miss, Master) del nombre
df["Title"] = df["Name"].apply(lambda n: n.split(",")[1].split(".")[0].strip())
df[["Name", "Title", "FamilySize", "IsAlone"]].head()

## 7. `groupby` · la operación más usada

`groupby` implementa el patrón **Split-Apply-Combine** formalizado por Hadley Wickham:

<img src="https://swcarpentry.github.io/r-novice-gapminder/fig/12-plyr-fig1.png" width="480" alt="Split-Apply-Combine">

1. **Split** — divide los datos en grupos según una o más columnas.
2. **Apply** — aplica una función a cada grupo (suma, promedio, conteo, función custom).
3. **Combine** — junta los resultados en una sola tabla.

Es el equivalente directo del `GROUP BY` de SQL.

In [ ]:
df.groupby(["Sex", "Pclass"], observed=True).agg(
    edad_media=("Age", "mean"),
    supervivencia=("Survived", "mean"),
    n=("Survived", "size"),
).round(2)

## 8. `merge` · joins entre tablas

`merge` implementa los 4 tipos de join de SQL:

- `how="inner"` — solo las filas con match en ambos (por defecto).
- `how="left"` — todas las izquierdas, con NaN donde no haya match.
- `how="right"` — espejo de left.
- `how="outer"` — todas de ambos lados.

`on="col"` indica la columna de unión; `left_on` / `right_on` si se llaman distinto en cada tabla.

In [ ]:
clases = pd.DataFrame({
    "Pclass": pd.Categorical([1, 2, 3]),
    "NombreClase": ["Primera", "Segunda", "Tercera"],
})
df = df.merge(clases, on="Pclass", how="left")
df[["Pclass", "NombreClase"]].head()

## 9. Exportar

In [ ]:
# Guardar el dataset limpio para las siguientes clases
# df.to_csv("data/processed/titanic_limpio.csv", index=False)
# df.to_parquet("data/processed/titanic_limpio.parquet")  # más eficiente
print("Dataset final:", df.shape)

---

## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso

- Identifiquen y decidan qué hacer con los nulos de su dataset (imputar, eliminar, mantener).
- Detecten y documenten outliers con IQR.
- Exporten el dataset limpio a `data/processed/`.

## 📦 Entregable del día

`data/processed/dataset_limpio.csv` + notebook documentando decisiones.

---

## 📚 Lecturas adicionales

Para profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.

### 🎬 Video recomendado

<iframe width="720" height="405" src="https://www.youtube.com/embed/QwgNjctlYnQ" title="Pandas desde cero #3 · Limpiar datos, valores nulos, eliminar datos" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

**[Pandas desde cero #3 · Limpiar datos, valores nulos, eliminar datos](https://www.youtube.com/watch?v=QwgNjctlYnQ)** · orvizar TV

_Tutorial visual en español sobre manejo de nulos, isnull/dropna/fillna y estrategias de imputación en datos sucios._